# Linear Regression using Pyspark

In [40]:
#create sparksession object
from pyspark.sql import SparkSession

#import Linear Regression from spark's MLlib
from pyspark.ml.regression import LinearRegression

#import corr function from pyspark functions // for the Pearson Correlation Coefficient for col1 and col2
from pyspark.sql.functions import corr

#import vectorassembler to create dense vectors
from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler

spark=SparkSession.builder.appName('lin_reg').getOrCreate()

In [41]:
#Load the dataset
df=spark.read.csv('../datasets/Linear_regression_dataset.csv',inferSchema=True,header=True)
df.show()

+-----+-----+-----+-----+-----+------+
|var_1|var_2|var_3|var_4|var_5|output|
+-----+-----+-----+-----+-----+------+
|  734|  688|   81|0.328|0.259| 0.418|
|  700|  600|   94| 0.32|0.247| 0.389|
|  712|  705|   93|0.311|0.247| 0.417|
|  734|  806|   69|0.315| 0.26| 0.415|
|  613|  759|   61|0.302| 0.24| 0.378|
|  748|  676|   85|0.318|0.255| 0.422|
|  669|  588|   97|0.315|0.251| 0.411|
|  667|  845|   68|0.324|0.251| 0.381|
|  758|  890|   64| 0.33|0.274| 0.436|
|  726|  670|   88|0.335|0.268| 0.422|
|  583|  794|   55|0.302|0.236| 0.371|
|  676|  746|   72|0.317|0.265|   0.4|
|  767|  699|   89|0.332|0.274| 0.433|
|  637|  597|   86|0.317|0.252| 0.374|
|  609|  724|   69|0.308|0.244| 0.382|
|  776|  733|   83|0.325|0.259| 0.437|
|  701|  832|   66|0.325| 0.26|  0.39|
|  650|  709|   74|0.316|0.249| 0.386|
|  804|  668|   95|0.337|0.265| 0.453|
|  713|  614|   94| 0.31|0.238| 0.404|
+-----+-----+-----+-----+-----+------+
only showing top 20 rows


In [42]:
#validate the size of data
print((df.count(), len(df.columns)))

(1232, 6)


In [43]:
df.describe().show(5,False)

+-------+-----------------+-----------------+------------------+--------------------+--------------------+-------------------+
|summary|var_1            |var_2            |var_3             |var_4               |var_5               |output             |
+-------+-----------------+-----------------+------------------+--------------------+--------------------+-------------------+
|count  |1232             |1232             |1232              |1232                |1232                |1232               |
|mean   |715.0819805194806|715.0819805194806|80.90422077922078 |0.3263311688311693  |0.25927272727272715 |0.39734172077922014|
|stddev |91.5342940441652 |93.07993263118064|11.458139049993724|0.015012772334166148|0.012907228928000298|0.03326689862173776|
|min    |463              |472              |40                |0.277               |0.214               |0.301              |
|max    |1009             |1103             |116               |0.373               |0.294               |0.491

In [44]:
#create the vector assembler inputCols → the list of columns you want to combine → ['var_1', 'var_2', 'var_3', 'var_4', 'var_5']
#                            outputCol → the name of the new column that will hold the combined vector → 'features' 

vec_assmebler=VectorAssembler(inputCols=['var_1', 'var_2', 'var_3', 'var_4', 'var_5'],outputCol='features')

In [45]:
#transform the values  The .transform(df) method applies the assembler to the DataFrame df. It creates a new column called 'features' that contains a dense vector of all the input column values for each row.
features_df=vec_assmebler.transform(df)

In [46]:
#validate the presence of dense vectors 
features_df.printSchema()

root
 |-- var_1: integer (nullable = true)
 |-- var_2: integer (nullable = true)
 |-- var_3: integer (nullable = true)
 |-- var_4: double (nullable = true)
 |-- var_5: double (nullable = true)
 |-- output: double (nullable = true)
 |-- features: vector (nullable = true)



In [47]:
#view the details of dense vector
features_df.show(5,False)

+-----+-----+-----+-----+-----+------+------------------------------+
|var_1|var_2|var_3|var_4|var_5|output|features                      |
+-----+-----+-----+-----+-----+------+------------------------------+
|734  |688  |81   |0.328|0.259|0.418 |[734.0,688.0,81.0,0.328,0.259]|
|700  |600  |94   |0.32 |0.247|0.389 |[700.0,600.0,94.0,0.32,0.247] |
|712  |705  |93   |0.311|0.247|0.417 |[712.0,705.0,93.0,0.311,0.247]|
|734  |806  |69   |0.315|0.26 |0.415 |[734.0,806.0,69.0,0.315,0.26] |
|613  |759  |61   |0.302|0.24 |0.378 |[613.0,759.0,61.0,0.302,0.24] |
+-----+-----+-----+-----+-----+------+------------------------------+
only showing top 5 rows


In [48]:
#create data containing input features and output column
model_df=features_df.select('features','output')

In [49]:
model_df.show(5,False)

+------------------------------+------+
|features                      |output|
+------------------------------+------+
|[734.0,688.0,81.0,0.328,0.259]|0.418 |
|[700.0,600.0,94.0,0.32,0.247] |0.389 |
|[712.0,705.0,93.0,0.311,0.247]|0.417 |
|[734.0,806.0,69.0,0.315,0.26] |0.415 |
|[613.0,759.0,61.0,0.302,0.24] |0.378 |
+------------------------------+------+
only showing top 5 rows


### Split Data - Train & Test sets


In [50]:
#split the data into 70/30 ratio for train test purpose
train_df,test_df=model_df.randomSplit([0.7,0.3])

## Build Linear Regression Model 

In [51]:
#Build Linear Regression model 
lin_Reg=LinearRegression(labelCol='output')

In [ ]:
#fit the linear regression model on training data set 
lr_model=lin_Reg.fit(train_df)

#That lr_model is your trained LinearRegressionModel. You’ll use it to make predictions.

25/11/14 05:38:26 WARN Instrumentation: [70403041] regParam is zero, which might cause numerical instability and overfitting.


In [53]:
lr_model.intercept

0.18640246531398735

In [54]:
print(lr_model.coefficients)

[0.0003428380281885661,5.091321872599616e-05,0.00017800287719423649,-0.6569969848290235,0.4985889658253749]


In [55]:
# After you train a linear regression model you can evaluate how well the model fits your training data by calling
# The returned training_predictions (actually a summary) has many useful attributes:
# Important: .evaluate() does not produce predictions
training_predictions=lr_model.evaluate(train_df)

In [ ]:
training_predictions.r2

0.8584632373778365

In [57]:
#make predictions on test data 
test_results=lr_model.evaluate(test_df)

In [58]:
#coefficient of determination value for model
test_results.r2

0.890536628037096

In [ ]:
predictions =lr_model.transform(test_df)

predictions.show(20,False) 

+--------------------+------+-------------------+
|            features|output|         prediction|
+--------------------+------+-------------------+
|[468.0,746.0,52.0...| 0.329|0.31902644992436735|
|[470.0,509.0,76.0...| 0.319|0.31178271908515587|
|[473.0,499.0,73.0...| 0.315| 0.3160268902978603|
|[486.0,610.0,61.0...| 0.332|0.31860809842774507|
|[501.0,774.0,51.0...| 0.315| 0.3285961383067715|
|[510.0,588.0,72.0...| 0.317| 0.3233919890856395|
|[511.0,576.0,76.0...| 0.329| 0.3290918558765252|
|[513.0,698.0,61.0...| 0.339|  0.330555870410055|
|[519.0,595.0,73.0...| 0.332| 0.3275338806222525|
|[522.0,621.0,72.0...| 0.317| 0.3275086418165659|
|[524.0,665.0,65.0...| 0.336| 0.3346028632541631|
|[544.0,551.0,82.0...| 0.344|0.33648721843823615|
|[548.0,770.0,60.0...| 0.351|0.35134565596399336|
|[552.0,683.0,71.0...| 0.335|0.34104763190911624|
|[556.0,674.0,62.0...| 0.348|0.34210122736762855|
|[558.0,740.0,60.0...|  0.36| 0.3479672988966525|
|[562.0,546.0,79.0...|  0.35|0.34266176831543593|


In [63]:
from pyspark.sql import Row
data = [
    Row(var_1=0.5, var_2=1.2, var_3=0.3, var_4=2.0, var_5=0.7),
    Row(var_1=0.8, var_2=0.5, var_3=1.5, var_4=0.4, var_5=1.1)
]
 
new_data_df = spark.createDataFrame(data)
new_test_df = vec_assmebler.transform(new_data_df)
new_test_df.show(2,False)

+-----+-----+-----+-----+-----+---------------------+
|var_1|var_2|var_3|var_4|var_5|features             |
+-----+-----+-----+-----+-----+---------------------+
|0.5  |1.2  |0.3  |2.0  |0.7  |[0.5,1.2,0.3,2.0,0.7]|
|0.8  |0.5  |1.5  |0.4  |1.1  |[0.8,0.5,1.5,0.4,1.1]|
+-----+-----+-----+-----+-----+---------------------+



In [65]:
new_predictions = lr_model.transform(new_test_df)
new_predictions.show()

+-----+-----+-----+-----+-----+--------------------+-------------------+
|var_1|var_2|var_3|var_4|var_5|            features|         prediction|
+-----+-----+-----+-----+-----+--------------------+-------------------+
|  0.5|  1.2|  0.3|  2.0|  0.7|[0.5,1.2,0.3,2.0,...|-0.7782933125265734|
|  0.8|  0.5|  1.5|  0.4|  1.1|[0.8,0.5,1.5,0.4,...| 0.4726182651379956|
+-----+-----+-----+-----+-----+--------------------+-------------------+

